# Stiffness 6 — section transitions

One inlet gradient, written two ways.

**As one section** carrying a linear coefficient: an exact ramp, and not a single interior section boundary.

**As a staircase** of $n$ constant steps: the same ramp to within a fraction of a percent once $n$ is large, and $n-1$ section boundaries along the way.

Pure transport again — no binding, no particle — so that nothing else can be blamed. Past about fifty steps the two produce the *same chromatogram*, so whatever the staircase costs is the price of having written the gradient that way, and nothing else.

The second knob is `INIT_STEP_SIZE`. CADET re-initialises the integrator at every section boundary, and this is the step size it restarts from.

In [ ]:
# This notebook lives one folder below the workshop root, while `utility`
# and everything under `data/` are addressed relative to that root.
# Step up into it before importing anything.
import os
import sys

root = os.path.abspath(os.curdir)
while not os.path.isdir(os.path.join(root, 'utility')):
    root, previous = os.path.dirname(root), root
    if root == previous:
        raise RuntimeError('no folder containing utility/ above this notebook')

os.chdir(root)
if root not in sys.path:
    sys.path.insert(0, root)

In [ ]:
import utility.section_explorer as explorer

explorer.show_setting()

## What to watch

The record reports the **deviation from the exact ramp**, so you can see when the staircase has stopped being an approximation and become the same answer — and the **extra steps per transition**, which is what one section boundary costs.

This is also the first study in the chapter where the **mean BDF order** is worth watching. Everywhere else it has sat at 5.00 and refused to move.

In [ ]:
from ipywidgets import interact

show_sections = explorer.make_explorer()

interact(show_sections, **show_sections.sliders);

## What happened

**The same answer costs thirteen times as much.**

| gradient written as | deviation from the ramp | BDF steps |
|---|---|---|
| one section with a slope | — | **963** |
| 50 step staircase | 0.017 % | 9074 |
| 200 step staircase | 0.001 % | **12831** |

At 200 steps the staircase and the ramp agree to one part in a hundred thousand. They are the same chromatogram. One of them takes thirteen times the work.

**And the order finally drops.** At 25 sections the mean BDF order falls to **4.53**, against 4.97 for the single ramp. Every transition throws away the order the integrator had built up and makes it start again at one. This is the only mechanism in the chapter that genuinely forces the integrator to a lower order — and it has nothing to do with eigenvalues.

**The cost splits into two parts, and only one is recoverable.** Measuring every step at 25 sections:

| `INIT_STEP_SIZE` | ramp steps per restart | total steps | mean order |
|---|---|---|---|
| 10⁻¹⁰ (default) | **28** | 6342 | 4.53 |
| 10⁻³ | **3** | 5885 | 4.73 |

Raising `INIT_STEP_SIZE` removes the climb out of the initial step size — 28 wasted steps per boundary become 3. What it cannot remove is the rest: each boundary introduces a jump in the inlet, and resolving that jump is real work. At 25 sections the restart is 28 steps of a 224 step bill.

So `INIT_STEP_SIZE` is worth knowing about, and it is not the fix. **The fix is not to have the transitions.**

<details>
<summary><b>Click to learn more: why the staircase is tempting</b></summary>

Writing a gradient as a list of constant concentrations is the natural thing to do when the gradient comes out of a method file, an experiment log, or another simulator. Each row becomes a section. Nobody writes two hundred sections on purpose, but a gradient sampled every thirty seconds over a two hour run is two hundred and forty of them.

`PIECEWISE_CUBIC_POLY` sections take a constant, linear, quadratic and cubic coefficient each. A linear gradient needs exactly one section with a linear coefficient. A curved one needs a handful. Almost no profile that occurs in practice needs more than a few, and the difference between a few and a few hundred is the factor of thirteen above.

</details>

> **Take-away.** Section boundaries are the one thing in this chapter that costs you a lot, costs it for a reason that is not stiffness, and is entirely within your control. Before tuning anything else about a slow gradient simulation, count your sections. If the number is large because a smooth profile was sampled into steps, replace it with the coefficients the profile actually has — same answer, a fraction of the work.
>
> *(Where this notebook belongs in the chapter is still open: the mechanism is consistent initialization and order restart rather than stiffness, even though it is measured here with the same instruments.)*